<a href="https://colab.research.google.com/github/dvarelaj/nlp-miniproyecto-icesi/blob/main/Sesion6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 1: Instalar dependencias

In [33]:
#pip install -U langchain langchain-community langchain-core
!pip install -U langchain-ollama langchain-chroma langchain-core langchain-text-splitters pypdf gradio -q
#!pip install -U langchain langchain-community langchain-chroma langchain-ollama -q

In [18]:
# Instalamos versiones específicas para minimizar conflictos en Colab
!pip install -U --force-reinstall langchain langchain-community langchain-chroma pypdf gradio -q

!pip install -U fsspec==2025.3.0  # Esto soluciona el error de datasets/gcsfs que viste

# Instalar zstd, requerido por Ollama para la extracción
!sudo apt-get update && sudo apt-get install zstd -y

# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import os

# Iniciar servidor
subprocess.Popen(['ollama', 'serve'], stdout=open(os.devnull, 'w'), stderr=open(os.devnull, 'w'))
time.sleep(15)

# Bajar modelo liviano pero potente
!ollama pull llama3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [19]:
!ollama list

NAME             ID              SIZE      MODIFIED               
llama3:latest    365c0bd3c000    4.7 GB    Less than a second ago    


## Paso 2: Carga de la Guía Médica de Sura

In [15]:
import os
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# URL de tu archivo en GitHub (usando el subdominio raw para descarga directa)
github_url = "https://raw.githubusercontent.com/dvarelaj/nlp-miniproyecto-icesi/main/Sesion%206/diabetes_mellitus.pdf"
pdf_path = "diabetes_mellitus_sura.pdf"

print("Descargando guía desde tu repositorio de GitHub...")
response = requests.get(github_url)

if response.status_code == 200:
    with open(pdf_path, "wb") as f:
        f.write(response.content)
    print("✅ Archivo descargado exitosamente.")
else:
    print(f"❌ Error: No se pudo acceder al archivo (Status: {response.status_code})")

# Cargar y dividir para el RAG
loader = PyPDFLoader(pdf_path)
docs = loader.load()

# Configuración de fragmentos (Chunking)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(f"📚 Documento procesado en {len(all_splits)} fragmentos.")

Descargando guía desde tu repositorio de GitHub...
✅ Archivo descargado exitosamente.
📚 Documento procesado en 30 fragmentos.


## Paso 3: Creación del Cerebro (Vector Store y RAG)

In [34]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configurar el LLM y Embeddings (Asegúrate de haber hecho el !ollama pull llama3)
llm = ChatOllama(model="llama3", temperature=0)
embeddings = OllamaEmbeddings(model="llama3")

# 2. Configurar el VectorStore (usando los 'all_splits' que ya tienes)
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Definir el Prompt con el contexto médico de Sura
template = """Eres un asistente médico experto de Sura.
Utiliza los siguientes fragmentos de contexto para responder la pregunta del usuario.
Si no conoces la respuesta basándote en el contexto, di que no lo sabes, no intentes inventar.

Contexto:
{context}

Pregunta: {question}

Respuesta profesional:"""

prompt = ChatPromptTemplate.from_template(template)

# 4. Crear la cadena usando LCEL (Esto reemplaza a create_retrieval_chain)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Prueba rápida:
# print(rag_chain.invoke("¿Cuál es el tratamiento inicial para la diabetes?"))

In [35]:
def ask_bot(pregunta):
    return rag_chain.invoke(pregunta)

# Luego lanzas Gradio con ask_bot como función

In [36]:
import gradio as gr

def responder_consulta(pregunta):
    # Invocamos la cadena RAG
    respuesta = rag_chain.invoke(pregunta)
    return respuesta

# Configuración visual
demo = gr.Interface(
    fn=responder_consulta,
    inputs=gr.Textbox(label="Consulta la Guía de Diabetes de Sura", placeholder="Ej: ¿Qué es la HbA1c?"),
    outputs=gr.Markdown(label="Respuesta del Asistente (RAG)"),
    title="SuraHealth-Bot 🤖",
    description="Asistente basado en Llama3 y LangChain para protocolos médicos."
)

demo.launch(share=True) # El 'share=True' es vital para que funcione el link público si lo necesitas

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c039a2c129139dc5d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
